In [1]:
!pip install navit-torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.5/533.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.6/241.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.2 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully unin

In [1]:
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
import torch.nn.functional as F
from navit.main import NaViT

from torchvision import datasets, transforms



# Assuming your unlabeled images for pretraining are in a folder structure
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])



In [2]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os
import random

from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os
import random



def mixup(data, targets, alpha=0.4):
    """
    Apply MixUp augmentation to a batch of images and labels.

    Parameters:
        data (torch.Tensor): Batch of images with shape (batch_size, channels, height, width).
        targets (torch.Tensor): Batch of labels with shape (batch_size,).
        alpha (float): MixUp parameter for Beta distribution.

    Returns:
        mixed_data (torch.Tensor): Augmented images.
        mixed_targets (tuple): Mixed targets (target_a, target_b, mix_ratio).
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = data.size(0)
    index = torch.randperm(batch_size)

    mixed_data = lam * data + (1 - lam) * data[index, :]
    target_a, target_b = targets, targets[index]
    return mixed_data, target_a, target_b, lam


def mixup_criterion(criterion, preds, target_a, target_b, lam):
    """
    Compute the MixUp loss.

    Parameters:
        criterion: Loss function (e.g., nn.CrossEntropyLoss).
        preds (torch.Tensor): Model predictions.
        target_a (torch.Tensor): First set of labels.
        target_b (torch.Tensor): Second set of labels.
        lam (float): MixUp ratio.

    Returns:
        loss (torch.Tensor): MixUp loss.
    """
    return lam * criterion(preds, target_a) + (1 - lam) * criterion(preds, target_b)

class CustomGroupedDatasetWithLabels(Dataset):
    def __init__(self, root_dir, group_sizes, transform=None):
        """
        root_dir: str, path to the dataset directory (organized into class folders)
        group_sizes: list of int, sizes of groups to create (e.g., [2, 2, 1])
        transform: torchvision.transforms, transformations to apply to the images
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []
        self.labels = []

        # Map folder names to numeric labels
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(os.listdir(root_dir))}

        # Collect all image paths and their labels
        for cls_name, label in self.class_to_idx.items():
            print(cls_name)
            print(label)
            class_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.endswith(('.png', '.jpg', '.jpeg')):
                        self.image_files.append(os.path.join(class_dir, img_name))

                        self.labels.append(label)

        # Shuffle images and labels together
        combined = list(zip(self.image_files, self.labels))
        random.shuffle(combined)
        self.image_files, self.labels = zip(*combined)

        # Create groups of images and labels based on group_sizes
        self.group_sizes = group_sizes
        self.groups = []
        idx = 0
        for size in group_sizes:
            group_files = self.image_files[idx:idx + size]
            group_labels = self.labels[idx:idx + size]
            self.groups.append((group_files, group_labels))
            idx += size

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        """
        Returns a tuple (images, labels), where:
          - images: List of tensors (C, H, W) for the group
          - labels: List of labels for the group
        """
        group_files, group_labels = self.groups[idx]
        images = []
        labels = []

        for img_path, label in zip(group_files, group_labels):
            img = Image.open(img_path).convert("RGB")  # Load image
            if self.transform:
                img = self.transform(img)  # Apply transformations
            images.append(img)
            labels.append(label)

        return images, labels



dataset_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"
group_sizes = [2, 2, 1]  # Example group sizes to mimic your input

# Create the dataset
grouped_dataset = CustomGroupedDatasetWithLabels(
    root_dir=dataset_dir,
    group_sizes=group_sizes,
    transform=transform)

#pretrain_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/sata_data', transform=transform)
#finetune_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/sata_data', transform=transform)


Docking
0
Non Docking
1
Dock-left
2
Dock-Right
3


In [3]:
import numpy as np
import torch

class EarlyStopping:

    def __init__(self, patience=7, verbose=False, delta=0.0, save_path="checkpoint.pt"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.save_path = save_path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Save model when validation loss decreases."""
        if self.verbose:
            print(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...")
        torch.save(model.state_dict(), self.save_path)
        self.val_loss_min = val_loss


In [3]:




grouped_loader = DataLoader(grouped_dataset, batch_size=1, shuffle=True)



batch_size = 1
#pretrain_loader = DataLoader(pretrain_dataset, batch_size=batch_size, shuffle=True)
#finetune_loader = DataLoader(finetune_dataset, batch_size=batch_size, shuffle=True)
def contrastive_loss(preds, temperature=0.5):
    """
    Computes the contrastive loss based on InfoNCE.

    Args:
    - preds (torch.Tensor): The embeddings from the model of shape (batch_size, embedding_dim).
    - temperature (float): Temperature parameter for scaling similarities.

    Returns:
    - loss (torch.Tensor): The computed contrastive loss.
    """
    # Normalize embeddings along the feature dimension
    preds = F.normalize(preds, dim=-1)

    # Compute similarity matrix
    similarity_matrix = torch.mm(preds, preds.t()) / temperature

    # Apply exp to get similarity scores
    exp_similarity_matrix = torch.exp(similarity_matrix)

    # Mask to zero out self-similarities (diagonal elements)
    mask = torch.eye(exp_similarity_matrix.shape[0], device=exp_similarity_matrix.device).bool()
    exp_similarity_matrix = exp_similarity_matrix * ~mask

    # Calculate contrastive loss
    positives = exp_similarity_matrix[mask].reshape(exp_similarity_matrix.shape[0], -1)
    negatives = exp_similarity_matrix.sum(dim=-1, keepdim=True) - positives

    # Compute the loss: -log of positive similarity over sum of positive and negatives
    loss = -torch.log(positives / (positives + negatives))
    loss = loss.mean()

    return loss
# Initialize model
n = NaViT(
    image_size=256,
    depth=4,
    patch_size=32,
    num_classes=4,  # Adjust as per specific task for finetuning
    dim=512,
    channels=3,
    heads=4,
    mlp_dim=512,
    dropout=0.6,
    emb_dropout=0.5,
    token_dropout_prob=0.2
)

# Specify device
device = 'cpu'
#n.load_state_dict(torch.load("/content/drive/MyDrive/navit.path"))
n.to(device)

# Define optimizer and hyperparameters
lr = 1e-4  # Adjust as needed
optimizer = torch.optim.Adam(n.parameters(), lr=1e-4, weight_decay=1e-5)
torch.nn.utils.clip_grad_norm_(n.parameters(), max_norm=1.0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

#optimizer = Adam(n.parameters(), lr=lr)
epochs = 15 # Number of training epochs for both pretraining and finetuning




# Pretraining loop (self-supervised learning)
def pretrain():
    n.train()
    for epoch in range(epochs):
        for i, (images, labels) in enumerate(grouped_loader):
            print(len(images))
            print(len(labels))
            labels=torch.Tensor(labels)




            preds = n.forward(images)

            # Define self-supervised loss, e.g., contrastive loss or other self-supervised technique
            # Assuming a dummy self-supervised loss calculation as a placeholder
            loss = contrastive_loss(preds)  # Define your self-supervised loss function

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print(f"Pretrain Epoch {epoch+1}/{epochs}, Step {i+1}, Loss: {loss.item():.4f}")

# Finetuning loop (supervised learning)
def finetune():
    n.train()
    for epoch in range(epochs):
        for i, (images, labels) in enumerate(grouped_loader):
            #images = [img.to(device) for img in images]
            images=[img.to(device) for img in images]
            labels=torch.Tensor(labels).long().to(device)
            preds = n(images).to(device)

            # Calculate loss (e.g., cross-entropy)
            loss = F.cross_entropy(preds, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step(loss)  # Adjust learning rate based on validation loss


            print(f"Finetune Epoch {epoch+1}/{epochs}, Step {i+1}, Loss: {loss.item():.4f}")

# Run pretraining and finetuning

#finetune()

In [7]:
# Take one batch from the DataLoader
batch = next(iter(grouped_loader))
images, labels = batch
images=[img.to(device) for img in images]
print(images[0].shape)
# Flatten the grouped images
#images = torch.cat(images, dim=0).to(device)  # Combine grouped images into a single batch
labels = torch.tensor(labels, dtype=torch.long).to(device)

# Test the forward pass
n.eval()  # Set model to evaluation mode
outputs = n(images)
print(f"Model Outputs Shape: {outputs.shape}")
print(f"Labels Shape: {labels.shape}")
print(f"Labels: {labels}")



torch.Size([1, 3, 256, 256])
Model Outputs Shape: torch.Size([2, 4])
Labels Shape: torch.Size([2])
Labels: tensor([3, 1])


In [ ]:
from torchvision import transforms

from torchvision import transforms
from PIL import Image
import torch

def predict_single_image(model, image_path, transform, device):
    """
    Test the model on a single image.

    Parameters:
        model: The trained NaViT model.
        image_path: Path to the image file.
        transform: Transformations to preprocess the image.
        device: The device to run the model on (CPU or GPU).

    Returns:
        Predicted class and confidence.
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")  # Ensure RGB format

    # Apply transformations and ensure image is divisible by the patch size (32)
    image = transform(image)  # Apply transformations
    image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device
    image=[image]

    # Forward pass through the model
    model.eval()
    with torch.no_grad():
        output = model(image)
        probs = torch.softmax(output, dim=1)  # Convert logits to probabilities
        predicted_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0, predicted_class].item()

    return predicted_class, confidence


# Define the same transformation as used during training
single_image_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Ensure the image is resized to 256x256
    transforms.ToTensor()  # Convert image to [C, H, W] format
])

# Path to your test image
image_path = "/content/drive/MyDrive/sata_data/Non Docking/IMG_8575.jpg"

# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=image_path,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")



Predicted Class: Dock-Right, Confidence: 0.41


In [ ]:
image_path = "/content/drive/MyDrive/sata_data/Dock-Right/IMG_8680.jpeg"

# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=image_path,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.41


In [ ]:
image_path = "/content/drive/MyDrive/sata_data/Dock-left/IMG_8669.jpeg"
img2="/content/drive/MyDrive/sata_data/Docking/IMG_8568.jpg"
# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=image_path,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.41


In [ ]:
img2="/content/drive/MyDrive/sata_data/Dock-left/IMG_8672.jpeg"
# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=img2,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.42


In [ ]:
img2="/content/drive/MyDrive/sata_data/Docking/IMG_8668.jpeg"
# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=img2,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.42


In [ ]:
img2="/content/drive/MyDrive/sata_data/Docking/IMG_8569.jpg"
# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=img2,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.41


In [ ]:
img2="/content/drive/MyDrive/sata_data/Docking/IMG_8571.jpg"
# Load and test on a single image
predicted_class, confidence = predict_single_image(
    model=n,  # The NaViT model
    image_path=img2,
    transform=single_image_transform,  # Transformation for preprocessing
    device='cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available
)

# Class labels (ensure they match your dataset)
class_names = ["Docking Side", "Non-Docking","Dock-Right","Dock-Left"]
print(f"Predicted Class: {class_names[predicted_class]}, Confidence: {confidence:.2f}")

Predicted Class: Dock-Right, Confidence: 0.42


In [ ]:
import os
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import random

# Define augmentation transformations
augmentations = transforms.Compose([
    transforms.RandomRotation(degrees=30),           # Random rotation
    #transforms.RandomErasing(p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),         # Horizontal flip
    transforms.RandomVerticalFlip(p=0.5),           # Vertical flip
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),  # Color jitter
    transforms.RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0)),  # Random crop and resize
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),      # Blur effect
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),         # Translation
    transforms.ToTensor()  # Convert to tensor
])

# Additional transformation to save the augmented images as files



In [8]:
def augment_dataset(input_dir, output_dir, num_augmentations=5):
    """
    Augment images from input_dir and save them to output_dir.

    Parameters:
        input_dir (str): Path to the folder containing original images (organized by class subfolders).
        output_dir (str): Path to save the augmented dataset.
        num_augmentations (int): Number of augmented images to generate per original image.

    Returns:
        None
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Loop through each class folder
    for class_folder in os.listdir(input_dir):
        class_input_path = os.path.join(input_dir, class_folder)
        class_output_path = os.path.join(output_dir, class_folder)

        if not os.path.exists(class_output_path):
            os.makedirs(class_output_path)

        # Loop through each image in the class folder
        if(os.path.exists(class_output_path) and  os.listdir(class_output_path)):
          continue

        for image_file in os.listdir(class_input_path):
            image_path = os.path.join(class_input_path, image_file)
            image = Image.open(image_path).convert("RGB")  # Ensure the image is in RGB

            # Save the original image to the output directory
            image.save(os.path.join(class_output_path, image_file))

            # Generate augmented images
            for i in range(num_augmentations):
                augmented_image = augmentations(image)
                augmented_image = to_pil(augmented_image)  # Convert back to PIL for saving

                # Save augmented image with a unique name
                augmented_image.save(os.path.join(class_output_path, f"{os.path.splitext(image_file)[0]}_aug_{i}.jpg"))

In [9]:
def featurize_dataset(input_dir, output_dir, num_augmentations=5):
    """
    Augment images from input_dir and save them to output_dir.

    Parameters:
        input_dir (str): Path to the folder containing original images (organized by class subfolders).
        output_dir (str): Path to save the augmented dataset.
        num_augmentations (int): Number of augmented images to generate per original image.

    Returns:
        None
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Loop through each class folder
    for class_folder in os.listdir(input_dir):
        class_input_path = os.path.join(input_dir, class_folder)
        class_output_path = os.path.join(output_dir, class_folder)

        if not os.path.exists(class_output_path):
            os.makedirs(class_output_path)

        # Loop through each image in the class folder
        if(os.path.exists(class_output_path) and  os.listdir(class_output_path)):
          continue

        for image_file in os.listdir(class_input_path):
            image_path = os.path.join(class_input_path, image_file)
            image = Image.open(image_path).convert("RGB")  # Ensure the image is in RGB

            # Save the original image to the output directory
            image.save(os.path.join(class_output_path, image_file))

            # Generate augmented images
            augmented_image = augmentations(image)
            augmented_image = to_pil(augmented_image)  # Convert back to PIL for saving

                # Save augmented image with a unique name
            augmented_image.save(os.path.join(class_output_path, f"{os.path.splitext(image_file)[0]}_feat_{i}.jpg"))

In [ ]:
# Apply augmentations and save the augmented dataset
import os
from torchvision.utils import save_image  # Import save_image
to_pil = transforms.ToPILImage()

def apply_and_save_augmentations(root_dir, output_dir, num_augmentations=500):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for class_name in os.listdir(root_dir):
        class_dir = os.path.join(root_dir, class_name)
        if os.path.isdir(class_dir):
            output_class_dir = os.path.join(output_dir, class_name)
            if not os.path.exists(output_class_dir):
                os.makedirs(output_class_dir)

            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_dir, img_name)
                    img = Image.open(img_path).convert("RGB")

                    for i in range(num_augmentations):
                        augmented_img = augmentations(img)
                        save_path = os.path.join(output_class_dir, f"{img_name.split('.')[0]}_aug_{i}.jpg")
                        save_image(augmented_img, save_path)

# Paths to the original and augmented datasets
root_dir = "/content/drive/MyDrive/sata_data/"
output_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"

# Apply augmentations and save the augmented dataset
#apply_and_save_augmentations(root_dir, output_dir, num_augmentations=5)

In [10]:
!pip install transformers

In [32]:
from transformers import SwinForImageClassification, AutoImageProcessor
# Load pretrained Swin Transformer
from transformers import SwinForImageClassification, AutoImageProcessor



In [33]:
from transformers import SwinForImageClassification, AutoImageProcessor
from torch.utils.data import DataLoader
import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from PIL import Image

# Load pretrained Swin Transformer
model_name = "microsoft/swin-tiny-patch4-window7-224"
model = SwinForImageClassification.from_pretrained(model_name, num_labels=4,ignore_mismatched_sizes=True)  # 4 classes
image_processor = AutoImageProcessor.from_pretrained(model_name)


Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
num_augmentations = 450  # Number of augmentations per image
#augment_dataset(root_dir, output_dir, num_augmentations)

In [34]:
import os
from PIL import Image
from transformers import SwinModel, AutoImageProcessor
import torch

# Load pretrained Swin Transformer (without the classification head)
model_name = "microsoft/swin-tiny-patch4-window7-224"
swin_model = SwinModel.from_pretrained("/content/drive/MyDrive/swin_finetuned2")

# Load the image processor
image_processor = AutoImageProcessor.from_pretrained(model_name)

# Move model to device
device =  "cpu"
swin_model.to(device)
# Example function to extract Swin features
def extract_swin_features(image_path, swin_model, image_processor):
    """
    Extract features from a Swin Transformer for a given image.
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    inputs = image_processor(image, return_tensors="pt").to(device)

    # Forward pass through Swin Transformer
    with torch.no_grad():
        outputs = swin_model(**inputs)

    # Extract features (last hidden state)
    features = outputs.last_hidden_state  # Shape: (batch_size, sequence_length, hidden_size)

    return features

def precompute_swin_features(dataset_dir, output_dir, swin_model, image_processor):
    """
    Precompute Swin features for all images in the dataset and save them.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for class_name in os.listdir(dataset_dir):
        class_dir = os.path.join(dataset_dir, class_name)
        if os.path.isdir(class_dir):
            output_class_dir = os.path.join(output_dir, class_name)
            if not os.path.exists(output_class_dir):
                os.makedirs(output_class_dir)

            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_dir, img_name)
                    features = extract_swin_features(img_path, swin_model, image_processor)

                    # Save features
                    save_path = os.path.join(output_class_dir, f"{img_name.split('.')[0]}.pt")
                    torch.save(features, save_path)

# Paths to the dataset and output directory
dataset_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive"
output_dir = "/content/drive/MyDrive/sata_data_swin_features_finetuned/"

# Precompute and save Swin features
#precompute_swin_features(dataset_dir, output_dir, swin_model, image_processor)

In [35]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset

class CustomGroupedDatasetWithFeatures(Dataset):
    def __init__(self, image_root_dir, feature_root_dir, group_sizes, transform=None):
        """
        Args:
            image_root_dir (str): Path to the dataset directory (organized into class folders).
            feature_root_dir (str): Path to the directory containing Swin features.
            group_sizes (list of int): Sizes of groups to create (e.g., [2, 2, 1]).
            transform (torchvision.transforms): Transformations to apply to the images.
        """
        self.image_root_dir = image_root_dir
        self.feature_root_dir = feature_root_dir
        self.transform = transform
        self.image_files = []
        self.feature_files = []
        self.labels = []

        # Map folder names to numeric labels
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(os.listdir(image_root_dir))}

        # Collect all image paths, feature paths, and their labels
        for cls_name, label in self.class_to_idx.items():
            image_class_dir = os.path.join(image_root_dir, cls_name)
            feature_class_dir = os.path.join(feature_root_dir, cls_name)

            if os.path.isdir(image_class_dir) and os.path.isdir(feature_class_dir):
                for img_name in os.listdir(image_class_dir):
                    if img_name.endswith(('.png', '.jpg', '.jpeg')):
                        # Image path
                        img_path = os.path.join(image_class_dir, img_name)
                        self.image_files.append(img_path)

                        # Feature path (assuming features are saved as .pt files with the same name)
                        feature_path = os.path.join(feature_class_dir, f"{img_name.split('.')[0]}.pt")
                        self.feature_files.append(feature_path)

                        # Label
                        self.labels.append(label)

        # Shuffle images, features, and labels together
        combined = list(zip(self.image_files, self.feature_files, self.labels))
        random.shuffle(combined)
        self.image_files, self.feature_files, self.labels = zip(*combined)

        # Create groups of images, features, and labels based on group_sizes
        self.group_sizes = group_sizes
        self.groups = []
        idx = 0
        for size in group_sizes:
            group_image_files = self.image_files[idx:idx + size]
            group_feature_files = self.feature_files[idx:idx + size]
            group_labels = self.labels[idx:idx + size]
            self.groups.append((group_image_files, group_feature_files, group_labels))
            idx += size

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        """
        Returns a tuple (images, features, labels), where:
          - images: List of tensors (C, H, W) for the group
          - features: List of tensors (feature_dim,) for the group
          - labels: List of labels for the group
        """
        group_image_files, group_feature_files, group_labels = self.groups[idx]
        images = []
        features = []
        labels = []

        for img_path, feature_path, label in zip(group_image_files, group_feature_files, group_labels):
            # Load image
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)  # Apply transformations
            images.append(img)

            # Load Swin features
            feature = torch.load(feature_path,map_location=torch.device('cpu'))  # Load precomputed Swin features
            features.append(feature)

            # Label
            labels.append(label)

        # Stack images and features into tensors
        images = torch.stack(images)  # Shape: (group_size, C, H, W)
        features = torch.stack(features)  # Shape: (group_size, feature_dim)
        labels = torch.tensor(labels)  # Shape: (group_size,)

        return images, features, labels

In [36]:
from torchvision import transforms

# Define transformations for the images
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize images
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
])

# Paths to the image and feature directories
image_root_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"
feature_root_dir = "/content/drive/MyDrive/sata_data_swin_features_finetuned/"

# Create the dataset
group_sizes = [2, 2, 1]  # Example group sizes
dataset = CustomGroupedDatasetWithFeatures(
    image_root_dir=image_root_dir,
    feature_root_dir=feature_root_dir,
    group_sizes=group_sizes,
    transform=transform
)

# Create a DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Iterate through the DataLoader
for images, features, labels in dataloader:
    print("Images shape:", images.shape)  # Shape: (batch_size, group_size, C, H, W)
    print("Features shape:", features.shape)  # Shape: (batch_size, group_size, feature_dim)
    print("Labels shape:", labels.shape)  # Shape: (batch_size, group_size)
    break

<ipython-input-35-6da25b377808>:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(feature_path,map_location=torch.device('cpu'))  # Load precomputed Swi

Images shape: torch.Size([1, 2, 3, 256, 256])
Features shape: torch.Size([1, 2, 1, 49, 768])
Labels shape: torch.Size([1, 2])


In [36]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from PIL import Image
from torchvision import transforms


from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from PIL import Image
import torch.nn as nn
# Define the NaViT model
"""
class NaViT(torch.nn.Module):
    def __init__(self, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.dim = dim  # Embedding dimension
        self.depth = depth  # Number of transformer layers
        self.heads = heads  # Number of attention heads
        self.mlp_dim = mlp_dim  # Dimension of the MLP in the transformer

        # Patch embedding layer (updated to accept 4 channels)
        self.patch_embedding = nn.Conv2d(
            in_channels=4,  # 3 channels for images + 1 channel for features
            out_channels=dim,
            kernel_size=patch_size,
            stride=patch_size
        )

        # Transformer encoder
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=heads,
                dim_feedforward=mlp_dim,
                dropout=dropout
            ),
            num_layers=depth
        )

        # Classification head
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, x):
        # Patch embedding
        x = self.patch_embedding(x)  # Shape: (batch_size, dim, num_patches_h, num_patches_w)
        x = x.flatten(2).transpose(1, 2)  # Shape: (batch_size, num_patches, dim)

        # Transformer encoder
        x = self.transformer(x)  # Shape: (batch_size, num_patches, dim)

        # Global average pooling
        x = x.mean(dim=1)  # Shape: (batch_size, dim)

        # Classification head
        x = self.classifier(x)  # Shape: (batch_size, num_classes)
        return x

# Define the NaViT model with Swin features
class NaViTWithFeatures(torch.nn.Module):
    def __init__(self, navit_model, feature_dim, image_height, image_width):
        super().__init__()
        self.navit_model = navit_model
        self.feature_projection = torch.nn.Linear(feature_dim, image_height * image_width)  # Project features to match spatial dimensions
        self.image_height = image_height
        self.image_width = image_width

    def forward(self, images, features):
        # Reshape images and features to handle group sizes
        batch_size, max_group_size, channels, height, width = images.shape
        images = images.view(batch_size * max_group_size, channels, height, width)  # Shape: (batch_size * max_group_size, channels, height, width)
        print("images shape",images.shape)

        # Check the shape of features
        print("Features shape:", features.shape)  # Debugging: Should be (batch_size, max_group_size, feature_dim)

        # Project features to match spatial dimensions of the images
        projected_features = self.feature_projection(features)  # Shape: (batch_size, max_group_size, height * width)
        print("Projected features shape:", projected_features.shape)  # Debugging: Should be (batch_size, max_group_size, height * width)

        # Reshape projected features to (batch_size * max_group_size, 1, height, width)
        projected_features = projected_features.view(batch_size * max_group_size, 1, height, width)  # Shape: (batch_size * max_group_size, 1, height, width)

        # Combine images and features (concatenate along channel dimension)
        combined_inputs = torch.cat([images, projected_features], dim=1)  # Shape: (batch_size * max_group_size, 4, height, width)

        # Pass through NaViT
        outputs = self.navit_model(combined_inputs)  # Shape: (batch_size * max_group_size, num_classes)

        # Reshape outputs back to (batch_size, max_group_size, num_classes)
        outputs = outputs.view(batch_size, max_group_size, -1)
        return outputs

# Define the dataset class
class CustomGroupedDatasetWithFeatures(Dataset):
    def __init__(self, image_root_dir, feature_root_dir, group_sizes, transform=None):
        self.image_root_dir = image_root_dir
        self.feature_root_dir = feature_root_dir
        self.transform = transform
        self.image_files = []
        self.feature_files = []
        self.labels = []

        # Map folder names to numeric labels
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(os.listdir(image_root_dir))}

        # Collect all image paths, feature paths, and their labels
        for cls_name, label in self.class_to_idx.items():
            image_class_dir = os.path.join(image_root_dir, cls_name)
            feature_class_dir = os.path.join(feature_root_dir, cls_name)

            if os.path.isdir(image_class_dir) and os.path.isdir(feature_class_dir):
                for img_name in os.listdir(image_class_dir):
                    if img_name.endswith(('.png', '.jpg', '.jpeg')):
                        # Image path
                        img_path = os.path.join(image_class_dir, img_name)
                        self.image_files.append(img_path)

                        # Feature path (assuming features are saved as .pt files with the same name)
                        feature_path = os.path.join(feature_class_dir, f"{img_name.split('.')[0]}.pt")
                        self.feature_files.append(feature_path)

                        # Label
                        self.labels.append(label)

        # Shuffle images, features, and labels together
        combined = list(zip(self.image_files, self.feature_files, self.labels))
        random.shuffle(combined)
        self.image_files, self.feature_files, self.labels = zip(*combined)

        # Create groups of images, features, and labels based on group_sizes
        self.group_sizes = group_sizes
        self.groups = []
        idx = 0
        for size in group_sizes:
            group_image_files = self.image_files[idx:idx + size]
            group_feature_files = self.feature_files[idx:idx + size]
            group_labels = self.labels[idx:idx + size]
            self.groups.append((group_image_files, group_feature_files, group_labels))
            idx += size

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        group_image_files, group_feature_files, group_labels = self.groups[idx]
        images = []
        features = []
        labels = []

        for img_path, feature_path, label in zip(group_image_files, group_feature_files, group_labels):
            # Load image and ensure it has 3 channels (RGB)
            img = Image.open(img_path).convert("RGB")  # Convert to RGB
            if self.transform:
                img = self.transform(img)  # Apply transformations
            images.append(img)

            # Load Swin features
            feature = torch.load(feature_path,map_location=torch.device('cpu'))  # Load precomputed Swin features
            feature = torch.mean(feature, dim=1)  # Aggregate over sequence dimension: (1, 49, 768) -> (1, 768)
            features.append(feature.squeeze(0))  # Remove batch dimension: (1, 768) -> (768,)

            # Label
            labels.append(label)

        # Stack images and features into tensors
        images = torch.stack(images)  # Shape: (group_size, C, H, W)
        features = torch.stack(features)  # Shape: (group_size, feature_dim)
        labels = torch.tensor(labels)  # Shape: (group_size,)

        return images, features, labels

# Define transformations for the images
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize images
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
])

# Paths to the image and feature directories
image_root_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"
feature_root_dir = "/content/drive/MyDrive/sata_data_swin_features_finetuned/"

# Create the dataset
group_sizes = [2, 2, 1]  # Example group sizes
dataset = CustomGroupedDatasetWithFeatures(
    image_root_dir=image_root_dir,
    feature_root_dir=feature_root_dir,
    group_sizes=group_sizes,
    transform=transform
)

# Collate function to handle variable group sizes
def collate_fn(batch):

    images, features, labels = zip(*batch)

    # Find the maximum group size in the batch
    max_group_size = max([img.shape[0] for img in images])

    # Pad images and features to the maximum group size
    padded_images = []
    padded_features = []
    padded_labels = []

    for img, feat, lbl in zip(images, features, labels):
        group_size = img.shape[0]

        # Pad images
        padding_size = max_group_size - group_size
        if padding_size > 0:
            img_padding = torch.zeros((padding_size, *img.shape[1:]), dtype=img.dtype)
            img = torch.cat([img, img_padding], dim=0)

        # Pad features
        if padding_size > 0:
            feat_padding = torch.zeros((padding_size, *feat.shape[1:]), dtype=feat.dtype)
            feat = torch.cat([feat, feat_padding], dim=0)

        # Pad labels
        if padding_size > 0:
            lbl_padding = torch.full((padding_size,), -1, dtype=lbl.dtype)  # Use -1 for padding
            lbl = torch.cat([lbl, lbl_padding], dim=0)

        padded_images.append(img)
        padded_features.append(feat)
        padded_labels.append(lbl)

    # Stack padded tensors
    padded_images = torch.stack(padded_images)  # Shape: (batch_size, max_group_size, C, H, W)
    padded_features = torch.stack(padded_features)  # Shape: (batch_size, max_group_size, feature_dim)
    padded_labels = torch.stack(padded_labels)  # Shape: (batch_size, max_group_size)

    return padded_images, padded_features, padded_labels

# Create a DataLoader with the collate_fn
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)



######################
feature_dim = 768  # Hidden size of Swin Transformer (e.g., 768 for Swin-Tiny)
image_height, image_width = 256, 256  # Spatial dimensions of the images
navit_with_swin = NaViTWithFeatures(model_navit, feature_dim, image_height, image_width).to(device)

# Define optimizer and loss function
optimizer = Adam(navit_with_swin.parameters(), lr=1e-4)
criterion = CrossEntropyLoss()

# Training loop
for epoch in range(20):  # Number of epochs
    navit_with_swin.train()
    for images, swin_features, labels in dataloader:
        images = images.to(device)
        swin_features = swin_features.to(device)
        labels = labels.to(device)

        # Forward pass through NaViT with combined inputs
        outputs = navit_with_swin(images, swin_features)  # Shape: (batch_size, max_group_size, num_classes)

        # Compute loss, ignoring padded labels
        mask = labels != -1  # Mask for valid labels
        loss = criterion(outputs[mask], labels[mask])

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

    """

'\nclass NaViT(torch.nn.Module):\n    def __init__(self, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, dropout=0.1):\n        super().__init__()\n        self.image_size = image_size\n        self.patch_size = patch_size\n        self.num_classes = num_classes\n        self.dim = dim  # Embedding dimension\n        self.depth = depth  # Number of transformer layers\n        self.heads = heads  # Number of attention heads\n        self.mlp_dim = mlp_dim  # Dimension of the MLP in the transformer\n\n        # Patch embedding layer (updated to accept 4 channels)\n        self.patch_embedding = nn.Conv2d(\n            in_channels=4,  # 3 channels for images + 1 channel for features\n            out_channels=dim,\n            kernel_size=patch_size,\n            stride=patch_size\n        )\n\n        # Transformer encoder\n        self.transformer = nn.TransformerEncoder(\n            nn.TransformerEncoderLayer(\n                d_model=dim,\n                nhead=heads,

In [37]:
mod=torch.load("/content/drive/MyDrive/navit.path",map_location=torch.device('cpu'))
# Initialize NaViT and the wrapper for Swin features
navit_model = NaViT(
    image_size=256,
    depth=8,
    patch_size=32,
    num_classes=4,  # Two classes: 'docking side' and 'non-docking'
    dim=1024,
    heads=16,
    mlp_dim=2048,
    dropout=0.1,

    token_dropout_prob=0.1
)
navit_model.load_state_dict(mod)

<ipython-input-37-df277a861ae2>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod=torch.load("/content/drive/MyDrive/navit.path",map_location=torch.device('cpu'))


<All keys matched successfully>

In [38]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# Custom Dataset Class
class CustomGroupedDatasetWithFeatures(Dataset):
    def __init__(self, image_root_dir, feature_root_dir, group_sizes, transform=None):
        self.image_root_dir = image_root_dir
        self.feature_root_dir = feature_root_dir
        self.transform = transform
        self.image_files = []
        self.feature_files = []
        self.labels = []

        # Map folder names to numeric labels
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(os.listdir(image_root_dir))}

        # Collect all image paths, feature paths, and their labels
        for cls_name, label in self.class_to_idx.items():
            image_class_dir = os.path.join(image_root_dir, cls_name)
            feature_class_dir = os.path.join(feature_root_dir, cls_name)

            if os.path.isdir(image_class_dir) and os.path.isdir(feature_class_dir):
                for img_name in os.listdir(image_class_dir):
                    if img_name.endswith(('.png', '.jpg', '.jpeg')):
                        # Image path
                        img_path = os.path.join(image_class_dir, img_name)
                        self.image_files.append(img_path)

                        # Feature path (assuming features are saved as .pt files with the same name)
                        feature_path = os.path.join(feature_class_dir, f"{img_name.split('.')[0]}.pt")
                        self.feature_files.append(feature_path)

                        # Label
                        self.labels.append(label)

        # Shuffle images, features, and labels together
        combined = list(zip(self.image_files, self.feature_files, self.labels))
        random.shuffle(combined)
        self.image_files, self.feature_files, self.labels = zip(*combined)

        # Create groups of images, features, and labels based on group_sizes
        self.group_sizes = group_sizes
        self.groups = []
        idx = 0
        for size in group_sizes:
            group_image_files = self.image_files[idx:idx + size]
            group_feature_files = self.feature_files[idx:idx + size]
            group_labels = self.labels[idx:idx + size]
            self.groups.append((group_image_files, group_feature_files, group_labels))
            idx += size

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        group_image_files, group_feature_files, group_labels = self.groups[idx]
        images = []
        features = []
        labels = []

        for img_path, feature_path, label in zip(group_image_files, group_feature_files, group_labels):
            # Load image
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            images.append(img)

            # Load Swin features
            feature = torch.load(feature_path, map_location=torch.device('cpu'))
            features.append(feature)

            # Label
            labels.append(label)

        # Stack images and features into tensors
        images = torch.stack(images)  # Shape: (group_size, C, H, W)
        features = torch.stack(features)  # Shape: (group_size, feature_dim)
        labels = torch.tensor(labels)  # Shape: (group_size,)

        return images, features, labels




In [42]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

image_root_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"
feature_root_dir = "/content/drive/MyDrive/sata_data_swin_features_finetuned/"
group_sizes = [2, 2, 1]

dataset = CustomGroupedDatasetWithFeatures(
        image_root_dir=image_root_dir,
        feature_root_dir=feature_root_dir,
        group_sizes=group_sizes,
        transform=transform
    )
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
num_classes = len(dataset.class_to_idx)


class NaViTWithSwin(nn.Module):
    def __init__(self, navit_model):
        super(NaViTWithSwin, self).__init__()
        self.navit_model = navit_model
        self.feature_projection = nn.Linear(768, 256)  # Example projection layer
        self.debug = True  # Set to False for training

    def forward(self, images, swin_features):
        batch_size, group_size, _, height, width = images.shape
        _, _, _, _, swin_dim = swin_features.shape

        # Reshape images for the NaViT input
        images = images.view(batch_size * group_size, 3, height, width)

        # Process Swin features
        swin_features = swin_features.view(batch_size * group_size, -1, swin_dim)  # Flatten
        swin_features = swin_features.mean(dim=1)  # Pool features

        # Project features and concatenate with images (if required)
        projected_features = self.feature_projection(swin_features)
        self
        if self.debug:
            print("Images reshaped for NaViT:", images.shape)
            print("Projected features shape:", projected_features.shape)

        # Combine projected features with images (or process separately)
        outputs = self.navit_model(images)

        return outputs

navit_with_swin = NaViTWithSwin(navit_model).to(device)

# Training loop
for epoch in range(epochs):
    navit_with_swin.train()
    total_loss = 0

    for images, swin_features, labels in dataloader:
        images, swin_features, labels = images.to(device), swin_features.to(device), labels.to(device)

        # Flatten group dimension for labels
        labels = labels.view(-1)

        # Forward pass
        outputs = navit_with_swin(images, swin_features)

        # Loss computation
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader)}")


<ipython-input-38-81bde2df442f>:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(feature_path, map_location=torch.device('cpu'))


Images reshaped for NaViT: torch.Size([1, 3, 256, 256])
Projected features shape: torch.Size([1, 256])


AssertionError: 

In [29]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms



# Dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

image_root_dir = "/content/drive/MyDrive/sata_data_augmented_aggressive/"
feature_root_dir = "/content/drive/MyDrive/sata_data_swin_features_finetuned/"
group_sizes = [2, 2, 1]

dataset = CustomGroupedDatasetWithFeatures(
    image_root_dir=image_root_dir,
    feature_root_dir=feature_root_dir,
    group_sizes=group_sizes,
    transform=transform
)

# Set the batch size to match the total group size
total_group_size = sum(group_sizes)
dataloader = DataLoader(dataset, batch_size=total_group_size, shuffle=True)

num_classes = 4  # Assuming two classes: docking and non-docking

# Define Model
class NaViTWithSwin(nn.Module):
    def __init__(self, navit_model):
        super(NaViTWithSwin, self).__init__()
        self.navit_model = navit_model
        self.feature_projection = nn.Linear(768, 256)  # Example projection layer
        self.debug = True  # Set to False for training

    def forward(self, images, swin_features):
        batch_size, group_size, _, height, width = images.shape
        _, _, _, _, swin_dim = swin_features.shape

        # Reshape images for the NaViT input
        images = images.view(batch_size * group_size, 3, height, width)

        # Process Swin features
        swin_features = swin_features.view(batch_size * group_size, -1, swin_dim)  # Flatten
        swin_features = swin_features.mean(dim=1)  # Pool features

        # Project features and concatenate with images (if required)
        projected_features = self.feature_projection(swin_features)

        if self.debug:
            print("Images reshaped for NaViT:", images.shape)
            print("Projected features shape:", projected_features.shape)

        # Combine projected features with images (or process separately)
        outputs = self.navit_model(images)

        return outputs

# Initialize Model
model_navit = ...  # Replace with your NaViT model initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
navit_with_swin = NaViTWithSwin(model_navit).to(device)

# Training Configurations
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(navit_with_swin.parameters(), lr=1e-4)
epochs = 10

# Training Loop
for epoch in range(epochs):
    navit_with_swin.train()
    total_loss = 0

    for images, swin_features, labels in dataloader:
        # Debug shapes
        print(f"Images shape: {images.shape}")
        print(f"Swin features shape: {swin_features.shape}")
        print(f"Labels shape: {labels.shape}")

        images, swin_features, labels = images.to(device), swin_features.to(device), labels.to(device)

        # Flatten group dimension for labels
        labels = labels.view(-1)

        # Forward pass
        outputs = navit_with_swin(images, swin_features)

        # Debug output
        print(f"Model outputs shape: {outputs.shape}")

        # Loss computation
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader)}")


<ipython-input-27-81bde2df442f>:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(feature_path, map_location=torch.device('cpu'))


RuntimeError: stack expects each tensor to be equal size, but got [2, 3, 256, 256] at entry 0 and [1, 3, 256, 256] at entry 1

In [ ]:
# Evaluation loop
navit_with_swin.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, swin_features, labels in dataloader:
        images = images.to(device)
        swin_features = swin_features.to(device)
        labels = labels.to(device)

        # Forward pass through NaViT with combined inputs
        outputs = navit_with_swin(images, swin_features)  # Shape: (batch_size, max_group_size, num_classes)

        # Compute accuracy, ignoring padded labels
        mask = labels != -1  # Mask for valid labels
        _, predicted = torch.max(outputs[mask], 1)
        total += labels[mask].size(0)
        correct += (predicted == labels[mask]).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")

<ipython-input-15-46f29fb7ed75>:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  feature = torch.load(feature_path)  # Load precomputed Swin features


Features shape: torch.Size([3, 2, 768])
Projected features shape: torch.Size([3, 2, 65536])
Validation Accuracy: 40.00%


In [23]:
import torch
from torch import nn
from PIL import Image
from torchvision import transforms
from transformers import SwinModel, AutoImageProcessor

# Define transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load pretrained Swin Transformer
swin_model_name = "microsoft/swin-tiny-patch4-window7-224"
swin_model = SwinModel.from_pretrained(swin_model_name)
image_processor = AutoImageProcessor.from_pretrained(swin_model_name)

# Define device
device = torch.device("cpu")
swin_model.to(device)

# Feature extractor
def extract_swin_features(image):
    inputs = image_processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = swin_model(**inputs)
        return outputs.last_hidden_state.mean(dim=1)  # Global average pooling

# NaViT model
class NaViTWithFeatures(nn.Module):
    def __init__(self, num_classes=4):
        super(NaViTWithFeatures, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.feature_proj = nn.Linear(768, 256 * 256)  # Project Swin features
        self.conv2 = nn.Conv2d(in_channels=64 + 1, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, images, features):
        x = self.conv1(images)
        projected_features = self.feature_proj(features).view(-1, 1, 256, 256)  # Reshape to spatial dimensions
        x = torch.cat([x, projected_features], dim=1)  # Combine image features with Swin features
        x = self.conv2(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

# Preprocess the image
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    normalized_image = transform(image).unsqueeze(0).to(device)
    return image, normalized_image

# Predict function
def predict(image_path):
    original_image, normalized_image = preprocess_image(image_path)

    # Extract features
    swin_features = extract_swin_features(original_image)

    # Combine inputs and perform prediction
    with torch.no_grad():
        outputs = model(normalized_image, swin_features)
        _, predicted = torch.max(outputs, 1)
    return predicted.item()

# Initialize and move NaViT model to device
model = NaViTWithFeatures(num_classes=4).to(device)

# Example usage
image_path = "/content/drive/MyDrive/sata_data/Docking/IMG_8571.jpg"
predicted_class = predict(image_path)
print(f"Predicted class: {predicted_class}")
# Map class index to class name
idx_to_class = {0: "Docking", 1: "Non Docking", 2: "Dock-left", 3: "Dock-Right"}
predicted_class_name = idx_to_class[predicted_class]
print(f"Predicted class: {predicted_class_name}")

Predicted class: 2
Predicted class: Dock-left


In [ ]:
mod=torch.load("/content/drive/MyDrive/navit.path",map_location=torch.device('cpu'))

<ipython-input-15-c4ebde423909>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod=torch.load("/content/drive/MyDrive/navit.path",map_location=torch.device('cpu'))


embedidings and won features direction

In [43]:
import torch
from torch.nn.functional import cosine_similarity
from torchvision import transforms
from PIL import Image
import os
from transformers import SwinForImageClassification, SwinConfig

# Load pretrained Swin model
swin_model = SwinForImageClassification.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
swin_model.eval()

# Define transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Function to extract embeddings
def extract_embedding(image_path, model, transform):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        features = model.forward_features(tensor).last_hidden_state.mean(dim=1)
    return features.squeeze(0)

# Load reference embeddings
reference_dir = "/path/to/reference/images/"
reference_embeddings = {}
for class_name in os.listdir(reference_dir):
    class_dir = os.path.join(reference_dir, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        embedding = extract_embedding(img_path, swin_model, transform)
        reference_embeddings[class_name] = embedding

# Classification based on closeness
def classify_image(image_path, model, transform, reference_embeddings):
    embedding = extract_embedding(image_path, model, transform)
    max_similarity = float('-inf')
    predicted_class = None

    for class_name, ref_embedding in reference_embeddings.items():
        similarity = cosine_similarity(embedding.unsqueeze(0), ref_embedding.unsqueeze(0)).item()
        if similarity > max_similarity:
            max_similarity = similarity
            predicted_class = class_name

    return predicted_class, max_similarity

# Test classification
test_image_path = "/path/to/test/image.jpg"
predicted_class, similarity = classify_image(test_image_path, swin_model, transform, reference_embeddings)
print(f"Predicted Class: {predicted_class}, Similarity: {similarity}")


FileNotFoundError: [Errno 2] No such file or directory: '/path/to/reference/images/'